# Variant Calling: FASTQ to VCF (Track 2 Capstone)

**BioNexus Hub - Track 2: Genomics & NGS**

Chain the whole NGS workflow into one mini-pipeline: align reads, sort, index, and call variants with bcftools. We plant a few known variants in a sample genome so you can check that the pipeline recovers exactly the right answers.

Run each cell in order with Shift+Enter.


## 1. Install the tools

`bwa` aligns, `samtools` sorts and indexes, `bcftools` calls variants.


In [ ]:
!apt-get -qq install -y bwa samtools bcftools
!bcftools --version | head -1

## 2. Build a reference and a sample with KNOWN variants

We make a reference genome, then copy it into a 'sample' and deliberately change a few bases at recorded positions. These planted SNPs are our answer key - a good pipeline should recover them.


In [ ]:
import random
random.seed(101)
bases = 'ACGT'

# Reference genome
ref = ''.join(random.choice(bases) for _ in range(5000))
with open('reference.fasta','w') as f:
    f.write('>chr1 demo reference\n')
    for i in range(0, len(ref), 60):
        f.write(ref[i:i+60] + '\n')

# Sample = reference with a few planted SNPs
sample = list(ref)
planted = []
for pos in [800, 1600, 2500, 3300, 4200]:
    old = sample[pos]
    new = random.choice([b for b in bases if b != old])
    sample[pos] = new
    planted.append((pos+1, old, new))   # 1-based position, like VCF
sample = ''.join(sample)

print('Planted variants (POS, REF, ALT):')
for p in planted: print(' ', p)

## 3. Generate reads from the SAMPLE genome

Reads come from the sample (which carries the variants), then get aligned back to the reference (which does not). Wherever the sample differs, the stacked reads will disagree with the reference - and that is exactly what the caller detects. We use deep coverage so every variant is well supported.


In [ ]:
L = 100
n_reads = 3000   # deep coverage so variants are well supported
with open('sample.fastq','w') as f:
    for n in range(n_reads):
        pos = random.randint(0, len(sample)-L)
        seq = list(sample[pos:pos+L])
        for j in range(L):
            if random.random() < 0.005:   # 0.5% sequencing error
                seq[j] = random.choice(bases)
        f.write(f'@read{n}\n{"".join(seq)}\n+\n{"I"*L}\n')
print(f'{n_reads} reads written from the sample genome')

## 4. Align, sort, index

Everything from the alignment and samtools lessons, in the standard three lines.


In [ ]:
!bwa index reference.fasta 2> /dev/null
!bwa mem reference.fasta sample.fastq 2> /dev/null | samtools sort -o sample.sorted.bam
!samtools index sample.sorted.bam
!echo '--- mapping summary ---'; samtools flagstat sample.sorted.bam | head -7

## 5. Call variants with bcftools

`mpileup` gathers the read bases stacked at every position; `call` decides which piles are real variants. `-v` keeps only variant sites.


In [ ]:
!bcftools mpileup -f reference.fasta sample.sorted.bam 2> /dev/null | \
  bcftools call -mv -Oz -o calls.vcf.gz
!bcftools index calls.vcf.gz
print('Variant calling done -> calls.vcf.gz')

## 6. Read the VCF and check it against the answer key

Each row is one variant: CHROM, POS, ID, REF, ALT, QUAL, ... Compare POS/REF/ALT to the variants we planted in step 2.


In [ ]:
!echo '--- called variants ---'
!bcftools view calls.vcf.gz | grep -v '^##' | cut -f1-6

In [ ]:
# Did we recover every planted variant?
import subprocess
out = subprocess.run(['bcftools','view','calls.vcf.gz'], capture_output=True, text=True).stdout
called = set()
for line in out.splitlines():
    if line.startswith('#'): continue
    c = line.split('\t')
    called.add((int(c[1]), c[3], c[4]))
print('Planted -> recovered?')
for pos, oldb, newb in planted:
    hit = any(p==pos for (p,r,a) in called)
    print(f'  pos {pos} {oldb}->{newb}: {"FOUND" if hit else "missed"}')

## You built a variant-calling pipeline

You went from raw reads to a VCF and verified it against a known answer. That's the core of clinical and research genomics.

### Your deliverable
1. Add a markdown cell at the top explaining each stage in your own words.
2. Report your mapping rate and the number of variants called.
3. Note what changes with real data (filtering by QUAL/DP, duplicate marking, a real reference).

**Try next:** lower `n_reads` to 200 and watch some variants drop out as coverage thins - proof that depth matters.

That completes Track 2. Next stop: **Track 3, Transcriptomics & RNA-seq.**
